# CoQA → Llama-3.1 judge (Google Colab)

End-to-end notebook for the **Beyond Flesch** project:

1. Download **stanfordnlp/coqa** (unique stories)
2. Run **meta-llama/Llama-3.1-8B-Instruct** with the same rubric as `step3f_llm_judge.py`
3. Build **`coqa_train.csv`** (clean format matching `outputs/clean_dataset/`)
4. Copy everything to **Google Drive**

## Before you run

- **Runtime → Change runtime type → GPU** (A100 or L4 recommended; T4 may OOM or be slow)
- Accept the [Llama 3.1 license](https://huggingface.co/meta-llama/Llama-3.1-8B-Instruct) on Hugging Face
- Create a [HF access token](https://huggingface.co/settings/tokens) (read access)

## Time (approx., 2000 stories)

| GPU | First run (download + judge) |
|-----|------------------------------|
| A100 | ~25–45 min |
| L4 | ~35–60 min |
| T4 | ~1–2.5 h (may need lower `GPU_MEM_FRAC` or `MAX_MODEL_LEN`) |

Set `MAX_STORIES = 500` for a quick smoke test (~10–15 min on A100 after model load).

## Output on Drive

`My Drive/beyond_flesch/coqa_judge/` — splits, judge CSV, clean `coqa_train.csv`, manifest

In [ ]:
# ── Edit these if needed ─────────────────────────────────────────────
MAX_STORIES = 2000          # unique CoQA stories (cap for cost/time)
MIN_CHARS = 80              # skip very short stories
SEED = 42

JUDGE_MODEL = "meta-llama/Llama-3.1-8B-Instruct"
MAX_MODEL_LEN = 2048
MAX_TOKENS = 8
GPU_MEM_FRAC = 0.90

# Local work dirs (Colab /content)
WORK_ROOT = "/content/coqa_judge_work"
SPLITS_DIR = f"{WORK_ROOT}/splits"
JUDGE_DIR = f"{WORK_ROOT}/llm_judge"
CLEAN_DIR = f"{WORK_ROOT}/clean_dataset"
HF_CACHE_DIR = f"{WORK_ROOT}/hf_cache"   # optional: copy weights to Drive

# Google Drive export folder
DRIVE_ROOT = "/content/drive/MyDrive/beyond_flesch/coqa_judge"

SPLIT_NAME = "coqa_train"
COQA_HF_DATASET = "stanfordnlp/coqa"

In [ ]:
# vLLM + HF stack (Colab-friendly pins)
!pip install -q "datasets>=2.18" pandas huggingface_hub
!pip install -q "transformers>=4.44" accelerate
!pip install -q vllm codecarbon

import json
import os
import shutil
import time

os.makedirs(WORK_ROOT, exist_ok=True)
os.makedirs(SPLITS_DIR, exist_ok=True)
os.makedirs(JUDGE_DIR, exist_ok=True)
os.makedirs(CLEAN_DIR, exist_ok=True)
os.makedirs(HF_CACHE_DIR, exist_ok=True)
os.environ["HF_HOME"] = HF_CACHE_DIR
os.environ["TRANSFORMERS_CACHE"] = HF_CACHE_DIR

print("Install done. WORK_ROOT =", WORK_ROOT)

In [ ]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError(
        "No GPU detected. In Colab: Runtime → Change runtime type → GPU (A100/L4)."
    )

name = torch.cuda.get_device_name(0)
mem_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"GPU: {name}")
print(f"VRAM: {mem_gb:.1f} GB")

if mem_gb < 14:
    print(
        "WARNING: <16 GB VRAM — if vLLM OOMs, set GPU_MEM_FRAC=0.85 and MAX_MODEL_LEN=1024 in config."
    )

In [ ]:
from huggingface_hub import login

# Paste your HF token when prompted (needs Llama 3.1 access)
login()
print("Hugging Face login OK")

In [ ]:
from google.colab import drive
import os

drive.mount("/content/drive")

for sub in ("splits", "llm_judge", "clean_dataset", "hf_cache", "logs"):
    os.makedirs(os.path.join(DRIVE_ROOT, sub), exist_ok=True)

print("Drive folder:", DRIVE_ROOT)
print("Subfolders: splits/, llm_judge/, clean_dataset/, hf_cache/, logs/")

## Step 1 — Download CoQA and build `coqa_train.csv` split

Deduplicates by story text (CoQA has many QA pairs per story). Saves to local disk and Drive `splits/`.

In [ ]:
import hashlib
import json
import os
import shutil
import time

import pandas as pd
from datasets import load_dataset

split_csv = f"{SPLITS_DIR}/{SPLIT_NAME}.csv"
drive_split_csv = f"{DRIVE_ROOT}/splits/{SPLIT_NAME}.csv"

if os.path.exists(split_csv) and os.path.getsize(split_csv) > 0:
    df_existing = pd.read_csv(split_csv)
    print(f"[prepare] Reusing existing split: {split_csv} ({len(df_existing)} rows)")
else:
    t0 = time.time()
    print(f"[prepare] Downloading {COQA_HF_DATASET} (train split)...")
    ds = load_dataset(COQA_HF_DATASET, split="train")
    print(f"[prepare] Loaded {len(ds)} CoQA rows (QA pairs); extracting unique stories...")

    rows = []
    seen = set()
    for ex in ds:
        story = str(ex.get("story") or "").strip()
        if len(story) < MIN_CHARS:
            continue
        key = hashlib.sha256(story.encode("utf-8")).hexdigest()
        if key in seen:
            continue
        seen.add(key)
        rows.append(
            {
                "full_text": story,
                "education_level": "middle",
                "source_dataset": "coqa",
                "domain": "reading",
                "label_source": "coqa_unlabeled",
                "subject": "coqa",
                "raw_label": "",
                "raw_grade": "",
                "source_tier": "silver",
                "label_confidence": 0.0,
                "sample_weight": 1.0,
                "split_group": f"coqa::{key[:16]}",
                "source_license": COQA_HF_DATASET,
                "split": SPLIT_NAME,
                "orig_split": SPLIT_NAME,
                "orig_idx": len(rows),
            }
        )
        if len(rows) >= MAX_STORIES:
            break

    if not rows:
        raise RuntimeError("No stories extracted — check MIN_CHARS / network.")

    df_existing = pd.DataFrame(rows).sample(frac=1.0, random_state=SEED).reset_index(drop=True)
    df_existing["orig_idx"] = range(len(df_existing))
    df_existing.to_csv(split_csv, index=False)
    elapsed = time.time() - t0
    print(f"[prepare] Wrote {len(df_existing)} unique stories -> {split_csv} ({elapsed:.1f}s)")

shutil.copy2(split_csv, drive_split_csv)
print(f"[prepare] Copied to Drive: {drive_split_csv}")
print(df_existing["full_text"].str.len().describe())
df_existing.head(2)

## Step 2 — Llama-3.1 judge (same rubric as project `step3f`)

Skips automatically if `coqa_train_judge.csv` already has the same row count as the split.

In [ ]:
import os
import re
import shutil
import time

import pandas as pd
from codecarbon import EmissionsTracker
from transformers import AutoTokenizer
from vllm import LLM, SamplingParams

RUBRIC = (
    "Classify the following text by its target reader's US education level.\n"
    "Choose exactly one of:\n"
    "- elementary  (US grades 1-5, simple vocabulary, short sentences)\n"
    "- middle      (US grades 6-8)\n"
    "- high        (US grades 9-12, advanced vocabulary, complex ideas)\n\n"
    "Text:\n{text}\n\n"
    "Reply with one word only: elementary, middle, or high."
)


def parse_label(raw):
    s = str(raw).strip().lower()
    s = re.sub(r"^[^a-z]+", "", s)
    if s.startswith("elem"):
        return "elementary"
    if s.startswith("mid"):
        return "middle"
    if s.startswith("high"):
        return "high"
    return "elementary"


split_csv = f"{SPLITS_DIR}/{SPLIT_NAME}.csv"
judge_csv = f"{JUDGE_DIR}/{SPLIT_NAME}_judge.csv"
drive_judge_csv = f"{DRIVE_ROOT}/llm_judge/{SPLIT_NAME}_judge.csv"

df = pd.read_csv(split_csv)
n = len(df)

if os.path.exists(judge_csv):
    done = pd.read_csv(judge_csv)
    if len(done) == n:
        print(f"[judge] Already complete ({n} rows) -> {judge_csv}")
        shutil.copy2(judge_csv, drive_judge_csv)
        print(f"[judge] Refreshed Drive copy: {drive_judge_csv}")
    else:
        print(f"[judge] Partial file ({len(done)}/{n}) — re-running judge...")
        os.remove(judge_csv)

if not (os.path.exists(judge_csv) and len(pd.read_csv(judge_csv)) == n):
    os.makedirs(f"{WORK_ROOT}/logs", exist_ok=True)
    tracker = EmissionsTracker(
        project_name="coqa_llama_judge_colab",
        output_dir=f"{WORK_ROOT}/logs",
        log_level="warning",
    )
    tracker.start()
    t0 = time.time()
    try:
        print(f"[judge] Loading {JUDGE_MODEL} (first run downloads ~16 GB)...")
        tokenizer = AutoTokenizer.from_pretrained(JUDGE_MODEL)
        llm = LLM(
            model=JUDGE_MODEL,
            dtype="bfloat16",
            enable_prefix_caching=True,
            gpu_memory_utilization=GPU_MEM_FRAC,
            max_model_len=MAX_MODEL_LEN,
            trust_remote_code=True,
        )
        sampling = SamplingParams(temperature=0.0, max_tokens=MAX_TOKENS)

        empty_msgs = [{"role": "user", "content": RUBRIC.format(text="")}]
        empty_prompt = tokenizer.apply_chat_template(
            empty_msgs, tokenize=False, add_generation_prompt=True
        )
        overhead = len(
            tokenizer(empty_prompt, add_special_tokens=False)["input_ids"]
        )
        budget = MAX_MODEL_LEN - overhead - MAX_TOKENS - 8
        if budget < 64:
            raise RuntimeError(
                f"Text budget too small: budget={budget} overhead={overhead}"
            )
        print(f"[judge] overhead={overhead} -> max_text_tokens={budget}")

        def build_prompt(text):
            ids = tokenizer(str(text), add_special_tokens=False)["input_ids"]
            if len(ids) > budget:
                text = tokenizer.decode(ids[:budget], skip_special_tokens=True)
            msgs = [{"role": "user", "content": RUBRIC.format(text=text)}]
            return tokenizer.apply_chat_template(
                msgs, tokenize=False, add_generation_prompt=True
            )

        prompts = [build_prompt(t) for t in df["full_text"].astype(str)]
        print(f"[judge] Judging {len(prompts)} stories...")
        outputs = llm.generate(prompts, sampling, use_tqdm=True)
        raws = [o.outputs[0].text for o in outputs]
        labels = [parse_label(r) for r in raws]

        rec = pd.DataFrame(
            {
                "orig_split": df["orig_split"] if "orig_split" in df.columns else SPLIT_NAME,
                "orig_idx": df["orig_idx"] if "orig_idx" in df.columns else df.index,
                "source_dataset": df["source_dataset"].astype(str),
                "education_level": df["education_level"].astype(str),
                "llm_judge_label": labels,
                "judge_raw_response": raws,
            }
        )
        rec.to_csv(judge_csv, index=False)
        elapsed = time.time() - t0
        agree = (rec["education_level"] == rec["llm_judge_label"]).mean()
        dist = dict(rec["llm_judge_label"].value_counts())
        print(f"[judge] Wrote {judge_csv} in {elapsed/60:.1f} min")
        print(f"[judge] agreement-with-placeholder-original={agree:.3f}")
        print(f"[judge] judge_dist={dist}")
    finally:
        tracker.stop()

    shutil.copy2(judge_csv, drive_judge_csv)
    print(f"[judge] Copied to Drive: {drive_judge_csv}")

pd.read_csv(judge_csv)["llm_judge_label"].value_counts()

## Step 3 — Build clean `coqa_train.csv` (ELECTRA-ready)

Same columns as `outputs/clean_dataset/train.csv`. Use **`education_level_judge`** as the training label.

In [ ]:
import json
import shutil

import pandas as pd

CORE_COLS = [
    "split",
    "orig_split",
    "orig_idx",
    "source_dataset",
    "subject",
    "raw_label",
    "full_text",
    "education_level_original",
    "education_level_judge",
    "judge_raw_response",
]

split_csv = f"{SPLITS_DIR}/{SPLIT_NAME}.csv"
judge_csv = f"{JUDGE_DIR}/{SPLIT_NAME}_judge.csv"
clean_csv = f"{CLEAN_DIR}/coqa_train.csv"
drive_clean_csv = f"{DRIVE_ROOT}/clean_dataset/coqa_train.csv"

sdf = pd.read_csv(split_csv)
jdf = pd.read_csv(judge_csv)
if len(sdf) != len(jdf):
    raise RuntimeError(f"Row mismatch: split={len(sdf)} judge={len(jdf)}")

out = pd.DataFrame()
out["split"] = [SPLIT_NAME] * len(sdf)
out["orig_split"] = sdf.get("orig_split", SPLIT_NAME)
out["orig_idx"] = sdf.get("orig_idx", range(len(sdf)))
out["source_dataset"] = "coqa"
out["subject"] = sdf["subject"].astype(str) if "subject" in sdf.columns else "coqa"
out["raw_label"] = sdf["raw_label"] if "raw_label" in sdf.columns else ""
out["full_text"] = sdf["full_text"].astype(str)
out["education_level_original"] = sdf["education_level"].astype(str)
out["education_level_judge"] = jdf["llm_judge_label"].astype(str)
out["judge_raw_response"] = jdf["judge_raw_response"].astype(str)
out = out[CORE_COLS]

out.to_csv(clean_csv, index=False)
shutil.copy2(clean_csv, drive_clean_csv)

manifest = {
    "split": SPLIT_NAME,
    "n_rows": int(len(out)),
    "judge_model": JUDGE_MODEL,
    "max_stories": MAX_STORIES,
    "seed": SEED,
    "coqa_dataset": COQA_HF_DATASET,
    "label_column_for_training": "education_level_judge",
    "judge_distribution": out["education_level_judge"].value_counts().to_dict(),
    "paths": {
        "local_clean": clean_csv,
        "drive_clean": drive_clean_csv,
        "drive_splits": f"{DRIVE_ROOT}/splits/{SPLIT_NAME}.csv",
        "drive_judge": f"{DRIVE_ROOT}/llm_judge/{SPLIT_NAME}_judge.csv",
    },
}
manifest_path = f"{DRIVE_ROOT}/clean_dataset/coqa_train_manifest.json"
with open(manifest_path, "w", encoding="utf-8") as f:
    json.dump(manifest, f, indent=2)

print(f"[clean] Wrote {len(out)} rows -> {clean_csv}")
print(f"[clean] Drive: {drive_clean_csv}")
print(f"[clean] Manifest: {manifest_path}")
out["education_level_judge"].value_counts()

## Step 4 — Sync all artifacts to Drive (optional HF cache)

Copies splits, judge output, clean CSV, emissions log, and optionally the Hugging Face model cache so you do not re-download on the next session.

In [ ]:
import glob
import json
import os
import shutil

COPY_HF_CACHE_TO_DRIVE = True   # set False to skip large model cache copy

def _copy_tree(src, dst):
    if not os.path.isdir(src):
        return 0
    os.makedirs(dst, exist_ok=True)
    n = 0
    for root, _, files in os.walk(src):
        rel = os.path.relpath(root, src)
        out_dir = os.path.join(dst, rel) if rel != "." else dst
        os.makedirs(out_dir, exist_ok=True)
        for fn in files:
            s = os.path.join(root, fn)
            d = os.path.join(out_dir, fn)
            if not os.path.exists(d) or os.path.getsize(s) != os.path.getsize(d):
                shutil.copy2(s, d)
                n += 1
    return n

pairs = [
    (f"{SPLITS_DIR}/{SPLIT_NAME}.csv", f"{DRIVE_ROOT}/splits/{SPLIT_NAME}.csv"),
    (f"{JUDGE_DIR}/{SPLIT_NAME}_judge.csv", f"{DRIVE_ROOT}/llm_judge/{SPLIT_NAME}_judge.csv"),
    (f"{CLEAN_DIR}/coqa_train.csv", f"{DRIVE_ROOT}/clean_dataset/coqa_train.csv"),
]
for src, dst in pairs:
    if os.path.exists(src):
        shutil.copy2(src, dst)
        print(f"OK {dst}")
    else:
        print(f"SKIP (missing) {src}")

for emissions in glob.glob(f"{WORK_ROOT}/logs/*.csv"):
    shutil.copy2(emissions, f"{DRIVE_ROOT}/logs/{os.path.basename(emissions)}")
    print(f"OK emissions {emissions}")

readme = f"""Beyond Flesch — CoQA Llama judge export
============================================
Generated by CoQA_Llama_Judge_Colab.ipynb

Use in ELECTRA notebook:
  CLEAN_DIR = "{DRIVE_ROOT}/clean_dataset"
  df_coqa = pd.read_csv(CLEAN_DIR + "/coqa_train.csv")
  df_train = pd.concat([df_train, df_coqa], ignore_index=True)

Training label column: education_level_judge
Domain column for DANN: source_dataset (value 'coqa')

Files:
  splits/coqa_train.csv       — raw stories before judge
  llm_judge/coqa_train_judge.csv
  clean_dataset/coqa_train.csv — merge this into train
  clean_dataset/coqa_train_manifest.json
  static_features/coqa_train_static.csv — static features (step3d)
"""
readme_path = f"{DRIVE_ROOT}/README.txt"
with open(readme_path, "w", encoding="utf-8") as f:
    f.write(readme)
print(f"Wrote {readme_path}")

if COPY_HF_CACHE_TO_DRIVE and os.path.isdir(HF_CACHE_DIR):
    print("Copying HF cache to Drive (may take several minutes)...")
    n = _copy_tree(HF_CACHE_DIR, f"{DRIVE_ROOT}/hf_cache")
    print(f"HF cache: copied/updated {n} files -> {DRIVE_ROOT}/hf_cache")
else:
    print("Skipped HF cache copy (COPY_HF_CACHE_TO_DRIVE=False or cache empty)")

print("\n=== DONE ===")
print("Drive root:", DRIVE_ROOT)
print("Main file for training:", f"{DRIVE_ROOT}/clean_dataset/coqa_train.csv")

## Next step — merge into `Electra_ScalarMix_DomainAdversarial.ipynb`

After this notebook finishes, in your ELECTRA Colab set:

```python
CLEAN_DIR = Path("/content/drive/MyDrive/beyond_flesch/coqa_judge/clean_dataset")
# or copy coqa_train.csv next to your existing clean_dataset folder

df_coqa = pd.read_csv(CLEAN_DIR / "coqa_train.csv")
df_train = pd.concat([df_train, df_coqa], ignore_index=True)
print(len(df_train), df_train["source_dataset"].value_counts())
```

Train on `education_level_judge`; use `source_dataset` for domain adversarial loss (CoQA → domain id for `coqa`).

## Step 5 — Static features (same as `step3d` / `train_static.csv`)

Writes `static_features/coqa_train_static.csv` on Drive (34 `stat_*` columns).

In [ ]:
!pip install -q textstat

import json
import sys
from pathlib import Path

# Repo scripts (adjust if your clone path differs)
SCRIPTS_DIR = Path("/content/Beyond-Flesch/logistic-regression/llm_as_a_judge/llm_as_a_judge/scripts")
if not SCRIPTS_DIR.exists():
    SCRIPTS_DIR = Path("/content/drive/MyDrive/beyond_flesch/scripts")
sys.path.insert(0, str(SCRIPTS_DIR))

from build_coqa_static import align_to_train_columns, compute_static_df  # noqa: E402

STATIC_DIR = Path("/content/drive/MyDrive/beyond_flesch/static_features")
STATIC_DIR.mkdir(parents=True, exist_ok=True)

coqa_csv = Path(DRIVE_ROOT) / "clean_dataset" / "coqa_train.csv"
df = pd.read_csv(coqa_csv)
out = compute_static_df(df["full_text"].astype(str).tolist(), desc="static/coqa_train")
out = align_to_train_columns(out, str(STATIC_DIR))
out_path = STATIC_DIR / "coqa_train_static.csv"
out.to_csv(out_path, index=False)
shutil.copy2(out_path, f"{DRIVE_ROOT}/static_features/coqa_train_static.csv")
print(f"Wrote {out_path}")
print(f"Copied -> {DRIVE_ROOT}/static_features/coqa_train_static.csv")